# Set up a customer-facing support agent on a governed Snowflake MCP server

This notebook creates and validates the **three tools** a customer-facing Gemini agent can reach through a single managed MCP server. The live, unscripted customer conversation runs in the Google agent UI — this notebook is the setup behind it.

**Why customer-facing is the ideal MCP use case:** when an external-facing agent talks to customers, you *must* strictly limit what it can touch. MCP earns its place when a customer asks unpredictable questions and the agent composes your governed tools on the fly. Schedule the predictable with Tasks; expose over MCP the ad-hoc customer conversations.

**Governance = two gates:**
1. **Declared tool surface** — the MCP spec exposes exactly 3 safe tools; nothing else is reachable.
2. **CUSTOMER_AGENT role** — the PAT is role-restricted; even if the tool list were tampered with, RBAC blocks SALES_FACT and arbitrary SQL.

Both gates must pass for anything to execute.

## The scoped tool surface

| Tool name | Snowflake object | Type |
|---|---|---|
| `search_reviews` | `MCP_HOL.SUPPORT.SEARCH_REVIEWS` | CORTEX_SEARCH_SERVICE_QUERY |
| `get_order_status` | `MCP_HOL.SUPPORT.GET_ORDER_STATUS` | GENERIC (function) |
| `file_ticket` | `MCP_HOL.SUPPORT.FILE_TICKET` | GENERIC (procedure) |

**Deliberately NOT exposed:** raw SQL execution, sales financials (SALES_FACT), other customers' orders, refunds/approvals (APPROVE_CASE), the fine-tuned classifier.

## Section 1 — The data behind the tools

Customer reviews (the corpus behind `search_reviews`) and the orders table (behind `get_order_status`).

In [ ]:
SELECT PRODUCT, RATING, REVIEW_TEXT, REVIEW_DATE
FROM MCP_HOL.SUPPORT.REVIEWS
WHERE PRODUCT = 'Summit Winter Jacket'
ORDER BY REVIEW_DATE DESC
LIMIT 10

In [ ]:
SELECT * FROM MCP_HOL.SUPPORT.ORDERS ORDER BY ORDER_DATE DESC

## Tool 1 — `search_reviews` (Cortex Search)

A Cortex Search service over the reviews table, exposed directly as a tool. The agent sends a natural-language query; the service returns semantically relevant reviews. No SQL, no table access — just a search API.

In [ ]:
-- Testing-only preview. The agent calls this semantically via the search_reviews MCP tool.
SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'MCP_HOL.SUPPORT.SEARCH_REVIEWS',
  '{"query":"zipper keeps breaking","columns":["REVIEW_TEXT","PRODUCT","RATING"],"limit":5}'
) AS search_results

## Tool 2 — `get_order_status` (scalar UDF)

A SQL UDF that returns a single human-readable line for ONE order. It runs with **owner's rights** — the CUSTOMER_AGENT role can *call* it but cannot read the underlying ORDERS table directly. The agent can only look up the order id the customer provides; it cannot scan all orders.

In [ ]:
SELECT MCP_HOL.SUPPORT.GET_ORDER_STATUS('ORD-10001') AS order_status

## Tool 3 — `file_ticket` (Jira via External Access)

A Python procedure that opens a Jira ticket for the customer's issue. It uses **External Access** (network rule + secret + integration) so the agent never sees credentials. On Jira failure it gracefully falls back — always records the ticket in Snowflake.

The full DDL (reference — not executed here to avoid creating junk tickets):

```sql
-- Network rule: allow egress to Jira
CREATE OR REPLACE NETWORK RULE MCP_HOL.SUPPORT.JIRA_RULE
  MODE=EGRESS TYPE=HOST_PORT VALUE_LIST=('your-domain.atlassian.net');

-- Secret: Jira API token (agent never sees this)
CREATE OR REPLACE SECRET MCP_HOL.SUPPORT.JIRA_CRED
  TYPE=PASSWORD USERNAME='agent@acme.com' PASSWORD='<api_token>';

-- Integration: binds rule + secret
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION JIRA_MCP_HOL_INT
  ALLOWED_NETWORK_RULES=(MCP_HOL.SUPPORT.JIRA_RULE)
  ALLOWED_AUTHENTICATION_SECRETS=(MCP_HOL.SUPPORT.JIRA_CRED) ENABLED=TRUE;

-- The procedure the tool wraps
CREATE OR REPLACE PROCEDURE MCP_HOL.SUPPORT.FILE_TICKET(ORDER_ID STRING, ISSUE STRING)
RETURNS STRING LANGUAGE PYTHON RUNTIME_VERSION=3.11 HANDLER='run'
EXTERNAL_ACCESS_INTEGRATIONS=(JIRA_MCP_HOL_INT)
PACKAGES=('snowflake-snowpark-python','requests')
SECRETS=('cred'=MCP_HOL.SUPPORT.JIRA_CRED)
AS $$
import _snowflake, requests
JIRA_HOST = 'your-domain.atlassian.net'
def run(session, order_id, issue):
    tid = 'SUP-' + str(session.sql('SELECT MCP_HOL.SUPPORT.TICKET_SEQ.NEXTVAL').collect()[0][0])
    ext = None
    try:
        c = _snowflake.get_username_password('cred')
        resp = requests.post(f'https://{JIRA_HOST}/rest/api/3/issue',
            auth=(c.username, c.password),
            json={'fields': {'project': {'key': 'SUP'},
                  'summary': f'Order {order_id}: {issue}',
                  'description': {'type':'doc','version':1,
                    'content':[{'type':'paragraph','content':[{'type':'text','text':issue}]}]},
                  'issuetype': {'name': 'Task'}}}, timeout=10)
        if resp.status_code in (200, 201): ext = resp.json().get('key')
    except Exception: ext = None
    ext_sql = 'NULL' if not ext else "'" + str(ext) + "'"
    session.sql('INSERT INTO MCP_HOL.SUPPORT.TICKETS '
      '(TICKET_ID,ORDER_ID,ISSUE,RECOMMENDED_REFUND,STATUS,EXTERNAL_CASE,CREATED_AT) '
      'VALUES (?,?,?,NULL,' + "'OPEN'," + ext_sql + ',CURRENT_TIMESTAMP())',
      params=[tid, order_id, issue]).collect()
    tail = f' Jira issue {ext} created.' if ext else ' (Jira not reachable yet - recorded in Snowflake.)'
    return f'Opened support ticket {tid} for order {order_id}.' + tail
$$;
```

## Build the tool surface

One `CREATE MCP SERVER` statement turns these three Snowflake objects into agent tools. Running the next cell **(re)deploys the live server** the Gemini agent connects to.

In [ ]:
CREATE OR REPLACE MCP SERVER MCP_HOL.AGENTS.MCP_HOL
FROM SPECIFICATION $$
tools:
  - name: "search_reviews"
    type: "CORTEX_SEARCH_SERVICE_QUERY"
    identifier: "MCP_HOL.SUPPORT.SEARCH_REVIEWS"
    title: "Search customer reviews"
    description: "Search customer product reviews for a topic (zipper problems, sizing, shipping, warmth, etc.) and return the most relevant matching reviews."
  - name: "get_order_status"
    type: "GENERIC"
    identifier: "MCP_HOL.SUPPORT.GET_ORDER_STATUS"
    title: "Look up order status"
    description: "Look up the status and shipment details of a SINGLE order by its order id. Returns the current status, carrier, tracking number, and delivery date."
    config:
      type: "function"
      warehouse: "COCO_WH"
      input_schema:
        type: "object"
        properties:
          order_id: { type: "string", description: "The customer's order id, e.g. ORD-10001" }
        required: ["order_id"]
  - name: "file_ticket"
    type: "GENERIC"
    identifier: "MCP_HOL.SUPPORT.FILE_TICKET"
    title: "Open a support ticket"
    description: "Open a support ticket for the customer's order describing their problem. Creates a Jira issue and records the ticket in Snowflake."
    config:
      type: "procedure"
      warehouse: "COCO_WH"
      input_schema:
        type: "object"
        properties:
          order_id: { type: "string", description: "The affected order id, e.g. ORD-10001" }
          issue: { type: "string", description: "Short description of the customer's problem" }
        required: ["order_id", "issue"]
$$;

## Scope it — the CUSTOMER_AGENT role (gate \#2)

The role gets USAGE on exactly the 3 tool objects + supporting infra (DB, schemas, warehouse, MCP server). Nothing else. Because the UDF and procedure are **owner's rights**, the role doesn't need SELECT on ORDERS or TICKETS — it can call the tools but cannot read the tables directly. And it has zero access to SALES schema, SALES_FACT, or any write/admin privileges.

In [ ]:
CREATE ROLE IF NOT EXISTS CUSTOMER_AGENT;
GRANT USAGE ON DATABASE MCP_HOL TO ROLE CUSTOMER_AGENT;
GRANT USAGE ON SCHEMA MCP_HOL.SUPPORT TO ROLE CUSTOMER_AGENT;
GRANT USAGE ON SCHEMA MCP_HOL.AGENTS TO ROLE CUSTOMER_AGENT;
GRANT USAGE ON WAREHOUSE COCO_WH TO ROLE CUSTOMER_AGENT;
GRANT USAGE ON CORTEX SEARCH SERVICE MCP_HOL.SUPPORT.SEARCH_REVIEWS TO ROLE CUSTOMER_AGENT;
GRANT USAGE ON FUNCTION MCP_HOL.SUPPORT.GET_ORDER_STATUS(VARCHAR) TO ROLE CUSTOMER_AGENT;
GRANT USAGE ON PROCEDURE MCP_HOL.SUPPORT.FILE_TICKET(VARCHAR,VARCHAR) TO ROLE CUSTOMER_AGENT;
GRANT USAGE ON MCP SERVER MCP_HOL.AGENTS.MCP_HOL TO ROLE CUSTOMER_AGENT;
GRANT ROLE CUSTOMER_AGENT TO USER ZHADA;

SHOW GRANTS TO ROLE CUSTOMER_AGENT;

## Verify the server and its tools

Confirm the server is live and see the three tools Gemini will discover via `tools/list`.

In [ ]:
SHOW MCP SERVERS IN SCHEMA MCP_HOL.AGENTS;

## Connect the Gemini agent

The agent authenticates with a **programmatic access token (PAT)** restricted to the `CUSTOMER_AGENT` role. The role restriction on the token IS gate \#2 — the managed MCP server runs every call under that role (no secondary roles), on the declared warehouse.

**MCP endpoint:**
```
https://sfsenorthamerica-zhada-aws1.snowflakecomputing.com/api/v2/databases/MCP_HOL/schemas/AGENTS/mcp-servers/MCP_HOL
```

**Google ADK wiring** (streamable-HTTP transport):
```python
from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool import MCPToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams

MCP_URL = "https://sfsenorthamerica-zhada-aws1.snowflakecomputing.com/api/v2/databases/MCP_HOL/schemas/AGENTS/mcp-servers/MCP_HOL"

agent = LlmAgent(
    model="gemini-3.1-pro-preview",   # served only from the 'global' Vertex location
    name="customer_support_agent",
    instruction="You are a friendly customer support assistant for an outdoor apparel retailer. "
                "Use the Snowflake tools to look up orders, search reviews, and file tickets. "
                "Never guess - always use the tools. Be concise and helpful.",
    tools=[MCPToolset(connection_params=StreamableHTTPConnectionParams(
        url=MCP_URL,
        headers={"Authorization": f"Bearer {SNOWFLAKE_PAT}"},
    ))],
)
```
ADK runs `tools/list` at startup; Gemini then calls the tools via `tools/call`.

In [ ]:
-- Issue the token. ROLE_RESTRICTION pins it to CUSTOMER_AGENT (gate #2).
-- Copy token_secret ONCE -> agent .env as SNOWFLAKE_PAT.
ALTER USER ZHADA ADD PROGRAMMATIC ACCESS TOKEN customer_agent_pat
  ROLE_RESTRICTION = 'CUSTOMER_AGENT'
  DAYS_TO_EXPIRY = 7;

## GO LIVE — in the Google agent UI

Paste this customer request and watch the agent chain the 3 tools:

> Hi — my order ORD-10001, the Summit Winter Jacket, the zipper broke the first time I wore it. Where's my order right now, is this a known problem, and can you help?

**Watch the agent compose:**

1. `get_order_status('ORD-10001')` → returns just that one order's delivery status and tracking
2. `search_reviews('zipper breaking')` → confirms it's a known defect across multiple reviews
3. `file_ticket('ORD-10001', 'Zipper broke on first wear - known defect per reviews')` → opens a Jira ticket and returns the reference

The agent **cannot**: see other customers' orders, run arbitrary SQL, view sales financials, or issue refunds — those tools were never exposed AND the CUSTOMER_AGENT role can't reach them.

## Recap

- **Customer-facing is the ideal MCP fit** — tight blast radius; you expose exactly what a customer conversation needs, nothing more.
- **One `CREATE MCP SERVER`** exposed 3 safe tools from existing Snowflake objects.
- **Two gates** — declared tool surface (only 3 tools in the spec) + CUSTOMER_AGENT role (RBAC blocks everything else even if the spec were tampered with).
- **`file_ticket` reaches Jira via External Access** without the agent or customer ever seeing credentials; graceful fallback logs locally.
- **The notebook is the setup, the live agent is the show** — we built and validated tools here; the unscripted customer conversation runs in Google's agent UI.

## Appendix A — The five MCP tool types

One MCP server can expose up to 50 tools. Every call runs under the **connecting token's role** (no secondary roles) on the declared warehouse.

| Type | What it exposes | Example |
|---|---|---|
| `CORTEX_SEARCH_SERVICE_QUERY` | Semantic search over a Cortex Search service | `search_reviews` |
| `CORTEX_ANALYST_MESSAGE` | Natural-language → SQL over a semantic view | sales Q&A |
| `SYSTEM_EXECUTE_SQL` | Run SQL directly (read-only or read-write) | ad-hoc queries |
| `GENERIC` | Wrap one UDF/procedure as a typed action | `get_order_status`, `file_ticket` |
| `CORTEX_AGENT_RUN` | Expose an entire Cortex Agent as one tool | nested agent |

**`GENERIC` config** — the safest action tool; one named operation, typed inputs, no freehand SQL:
- `type`: `function` (scalar UDF) or `procedure`
- `input_schema`: the typed args the agent must fill
- `warehouse` / `query_timeout`: compute + limit

**`SYSTEM_EXECUTE_SQL` config** — powerful, scope deliberately:
- `read_only: true` = SELECT only (default) · `false` = writes + DDL
- `query_timeout`: max seconds per query
- `warehouse`: compute to run on

## Appendix B — Full Jira External Access wiring

The complete DDL to wire the `file_ticket` tool to a real Jira instance. Replace the placeholder domain and API token with real values. Note: `APPROVE_CASE` and refund logic are deliberately NOT exposed over MCP.

```sql
-- 1) Network rule: allow egress to your Jira host
CREATE OR REPLACE NETWORK RULE MCP_HOL.SUPPORT.JIRA_RULE
  MODE=EGRESS TYPE=HOST_PORT VALUE_LIST=('your-domain.atlassian.net');

-- 2) Secret: Jira API token (the agent never sees this)
CREATE OR REPLACE SECRET MCP_HOL.SUPPORT.JIRA_CRED
  TYPE=PASSWORD USERNAME='agent@acme.com' PASSWORD='REPLACE_WITH_JIRA_API_TOKEN';

-- 3) External Access Integration: binds rule + secret
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION JIRA_MCP_HOL_INT
  ALLOWED_NETWORK_RULES=(MCP_HOL.SUPPORT.JIRA_RULE)
  ALLOWED_AUTHENTICATION_SECRETS=(MCP_HOL.SUPPORT.JIRA_CRED)
  ENABLED=TRUE;

-- 4) The procedure (owner's rights, graceful Jira fallback)
CREATE OR REPLACE PROCEDURE MCP_HOL.SUPPORT.FILE_TICKET(ORDER_ID STRING, ISSUE STRING)
RETURNS STRING LANGUAGE PYTHON RUNTIME_VERSION=3.11 HANDLER='run'
EXTERNAL_ACCESS_INTEGRATIONS=(JIRA_MCP_HOL_INT)
PACKAGES=('snowflake-snowpark-python','requests')
SECRETS=('cred'=MCP_HOL.SUPPORT.JIRA_CRED)
EXECUTE AS OWNER
AS $$
import _snowflake, requests
JIRA_HOST = 'your-domain.atlassian.net'
def run(session, order_id, issue):
    tid = 'SUP-' + str(session.sql('SELECT MCP_HOL.SUPPORT.TICKET_SEQ.NEXTVAL').collect()[0][0])
    ext = None
    try:
        c = _snowflake.get_username_password('cred')
        resp = requests.post(f'https://{JIRA_HOST}/rest/api/3/issue',
            auth=(c.username, c.password),
            json={'fields': {'project': {'key': 'SUP'},
                  'summary': f'Order {order_id}: {issue}',
                  'description': {'type':'doc','version':1,
                    'content':[{'type':'paragraph','content':[{'type':'text','text':issue}]}]},
                  'issuetype': {'name': 'Task'}}}, timeout=10)
        if resp.status_code in (200, 201): ext = resp.json().get('key')
    except Exception: ext = None
    ext_sql = 'NULL' if not ext else "'" + str(ext) + "'"
    session.sql('INSERT INTO MCP_HOL.SUPPORT.TICKETS '
      '(TICKET_ID,ORDER_ID,ISSUE,RECOMMENDED_REFUND,STATUS,EXTERNAL_CASE,CREATED_AT) '
      'VALUES (?,?,?,NULL,' + "'OPEN'," + ext_sql + ',CURRENT_TIMESTAMP())',
      params=[tid, order_id, issue]).collect()
    tail = f' Jira issue {ext} created.' if ext else ' (Jira not reachable yet - recorded in Snowflake.)'
    return f'Opened support ticket {tid} for order {order_id}.' + tail
$$;
```